In [2]:
#### mv utils ####

import modules.data as d
import modules.model as m
import modules.train as t
import modules.utils as u

import torch
import torch.nn as nn
from pathlib import Path

#### packages ####

import numpy as np
import torch.nn.functional as F
from modules.model import _get_edge_index

#### typing ####

from torch import Tensor
from pandas import DataFrame
from typing import Literal

In [ ]:
# dataset dir
datasets = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')

# get device
device, generator = u.Devices().auto_set_device()

# get data
data = d.TCGAData(
    # datasets
    tcga_project = 'TCGA-BRCA',
    metadata_subtype_col = 'paper_BRCA_Subtype_PAM50',

    # dirs
    tcga_dir = datasets / 'tcga',
    relation_filepath = datasets / 'other/relation_ohe.csv',
    
    # col, filter
    y_col = 'subtype',
    drop = {'subtype':['Normal', 'Primary Tumor']},
)


*** Device() ***
device = cuda:0

**** Data() ****
log0_method      log1p            str
class_weights    (6,)             Tensor (cuda:0)
gene_counts      (4383, 1179)     DataFrame
metadata         (1179, 3)        DataFrame
relation         (32798, 18)      DataFrame
node_id_map      4383             dict
masks            305              list
X                (1179, 4383, 1)  Tensor (cuda:0)
y                (1179, 6)        Tensor (cuda:0)
y_labels         6                list
num_samples      1179             int
num_nodes        4383             int
num_features     1                int
num_labels       6                int
num_masks        305              int



---

In [4]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax

class GATv2Layer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(node_dim=0, aggr='add')  # Aggregate messages by summation
        self.W = torch.nn.Linear(in_channels, out_channels, bias=False)
        self.attn = torch.nn.Parameter(torch.Tensor(1, 2 * out_channels))

        torch.nn.init.xavier_uniform_(self.W.weight)
        torch.nn.init.xavier_uniform_(self.attn)

    def forward(self, x, edge_index):
        # Linear transformation of node features
        x_transformed = self.W(x)  # [num_nodes, out_channels]

        # Start propagating messages
        return self.propagate(edge_index, x=x_transformed)

    def message(self, x_i, x_j, edge_index_i):
        # x_i: features of target nodes [num_edges, out_channels]
        # x_j: features of source nodes [num_edges, out_channels]

        # Compute attention coefficients using concatenation and nonlinearity (GATv2)
        alpha = torch.cat([x_i, x_j], dim=-1)  # [num_edges, 2 * out_channels]
        alpha = F.leaky_relu(alpha, negative_slope=0.2)
        alpha = (alpha @ self.attn.T)  # [num_edges, 1]

        # Normalize coefficients using softmax over edges of the target node
        alpha = softmax(alpha, edge_index_i)  # [num_edges, 1]

        # Multiply source node features by attention coefficients
        return alpha * x_j  # [num_edges, out_channels]


In [11]:
# format data to work in PyG
x = data.X[:1,:].squeeze(0)
edge_index = _get_edge_index(data.relation, method='in').to(device)

# define layer
layer = GATv2Layer(
    in_channels=data.num_features,
    out_channels=data.num_labels
).to(device)

# test forward pass: 
print(x.shape)
print(layer(x, edge_index).shape)

torch.Size([4383, 1])
torch.Size([4383, 6])


---

In [14]:
print(data)

log0_method      log1p            str
class_weights    (6,)             Tensor (cuda:0)
gene_counts      (4383, 1179)     DataFrame
metadata         (1179, 3)        DataFrame
relation         (32798, 18)      DataFrame
node_id_map      4383             dict
masks            305              list
X                (1179, 4383, 1)  Tensor (cuda:0)
y                (1179, 6)        Tensor (cuda:0)
y_labels         6                list
num_samples      1179             int
num_nodes        4383             int
num_features     1                int
num_labels       6                int
num_masks        305              int

